In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import fnmatch

In [21]:
def get_averages(path, filename, column_niter=0, column_conv=1, column_avn=2, 
                 column_runtime=10, skip=0, val_col_conv="1"):
    fin = open(f"{path}/{filename}", "r")
    av_niter = 0.0
    nconv = 0
    nsamples_runtime = 0
    av_avn = 0.0
    av_avn_sqr = 0.0
    av_runtime = 0.0
    for _ in range(skip):
        fin.readline()
    while True:
        line = fin.readline()
        if not line:
            break
        line = line.split()
        if line[column_conv] == val_col_conv:
            nconv += 1
            av_niter += float(line[column_niter])
            av_avn += float(line[column_avn])
            av_avn_sqr += float(line[column_avn])**2
            if float(line[column_runtime]) < 1e6:
                av_runtime += float(line[column_runtime])
                nsamples_runtime += 1
    fin.close()
    if nconv > 0:
        av_niter /= nconv
        av_avn /= nconv
        av_avn_sqr /= nconv
        av_runtime /= nsamples_runtime if nsamples_runtime > 0 else 0
    error_avn = np.sqrt(abs(av_avn_sqr - av_avn**2))
    return av_niter, av_avn, error_avn, av_runtime, nconv

In [7]:
def filter_files(path, pattern, pos_sigma=7):
    files = []
    # Iterate through the files in the specified directory
    for filename in os.listdir(path):
        if fnmatch.fnmatch(filename, pattern):
            parts = filename.split('_')
            files.append((filename, float(parts[pos_sigma])))
    sorted_pairs = sorted(files, key=lambda x: x[1])  # Sort by sigma value
    return sorted_pairs

In [14]:
def get_all_averages(path, pattern, pos_sigma=16, column_niter=0, column_conv=1, column_avn=2, 
                     column_runtime=10, skip=0, val_col_conv="1"):
    sorted_files = filter_files(path, pattern, pos_sigma=pos_sigma)
    data = []
    for filename, sigma in sorted_files:
        av_niter, av_avn, error_avn, av_runtime, nconv = get_averages(path, filename, 
                                column_niter=column_niter, column_conv=column_conv, column_avn=column_avn,
                                column_runtime=column_runtime, skip=skip, val_col_conv=val_col_conv)
        data.append((sigma, av_niter, av_avn, error_avn, av_runtime, nconv))
    return np.array(data)

In [5]:
def print_averages(pathout, fileout, data, header, ndigits_first=3, ndigits_rest=6):
    fout = open(f"{pathout}/{fileout}", "w")
    fout.write(f"{header}\n")
    for row in data:
        fout.write(f"{row[0]:.{ndigits_first}f}")
        for item in row[1:]:
            fout.write(f"\t{item:.{ndigits_rest}f}")
        fout.write(f"\n")
    fout.close()

In [22]:
path_in = "/mnt/d/Research/Ecology/Results/IBMF/for_average_abundance/AllData"
pattern = "IBMF_seq_eps_0.000_mu_0.270_sigma_*_N_1024_c_3_T_0.000_lambda_0.000_PD_Lotka_Volterra_final_av0_0.5_dn_0.5_ninitconds_1000_tol_1e-6_maxiter_10000_damping_0.2_nseq_1.txt"
path_out = "/mnt/d/Research/Ecology/Results/IBMF/for_average_abundance/"
file_out = "IBMF_seq_eps_0.000_mu_0.270_N_1024_c_3_T_0.000_lambda_0.000_PD_Lotka_Volterra_final_av0_0.5_dn_0.5_ninitconds_1000_tol_1e-6_maxiter_10000_damping_0.2_nseq_1_averages.txt"
pos_sigma = 7
column_niter = 0
column_conv = 1
column_avn = 2
column_runtime = 10
skip = 0
val_col_conv = "1"
header = "sigma\tav_niter\tav_avn\tstd_avn\tav_runtime\tnconv"
data = get_all_averages(path_in, pattern, pos_sigma=pos_sigma, column_niter=column_niter, column_conv=column_conv, column_avn=column_avn, column_runtime=column_runtime, skip=skip, val_col_conv=val_col_conv)
print_averages(path_out, file_out, data, header)

In [23]:
path_in = "/mnt/d/Research/Ecology/Langevin/Results/for_average_abundance/Results"
pattern = "Measures_Summary_GLV_epsilon_0.000_Partially_AsymGauss_lambda_1e-06_tol_1e-08_N_1024_c_3.00_mu_0.270_sigma_*_T_0.000_Ext_1.txt"
path_out = "/mnt/d/Research/Ecology/Langevin/Results/for_average_abundance/"
file_out = "Langevin_epsilon_0.000_Partially_AsymGauss_lambda_1e-06_tol_1e-08_N_1024_c_3.00_mu_0.270_T_0.000_Ext_1.txt"
pos_sigma = 18
column_niter = 2
column_conv = 6
val_col_conv = "0"
column_avn = 8
column_runtime = 7
skip = 1
header = "sigma\tav_tfinal\tav_avn\tstd_avn\tav_runtime\tnconv"
data = get_all_averages(path_in, pattern, pos_sigma=pos_sigma, column_niter=column_niter, column_conv=column_conv, column_avn=column_avn, column_runtime=column_runtime, skip=skip, val_col_conv=val_col_conv)
print_averages(path_out, file_out, data, header)